# Stage 3.B — distributed aero-first BO campaign (one session)

Runs **one async session** of the corrected 3D campaign (`Blade3DObjective`, cycle-mean CFz, N_RADIAL=40 — ADR-0004). Open this notebook in each Colab session (3 total) and set only `SESSION_INDEX` (0/1/2). All sessions share one Drive ledger and coordinate through it **without communicating**: each reads every session's results, refits the GP on the combined data, and claims non-overlapping designs (atomic markers) so none is run twice.

**Async, not batched:** the instant one eval finishes, a new design is dispatched to that freed worker (conditioned on all in-flight work) — no worker ever idles waiting for a batch. Cell 5 **validates** this actually happened from the ledger telemetry.

**Resumable:** every evaluation is appended to the shared ledger, so re-running the Run cell after a Colab drop resumes with no lost work. Orchestration only — the loop lives in `scripts/run_blade_campaign_distributed.py` + `fanopt.bo.distributed_campaign`.

## 1. Repo + deps + SU2 + Drive

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")   # 1 thread/worker -> N processes on N cores

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "main"  # the Stage-3 campaign machinery is merged to main
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gmsh", "cadquery"], check=True)
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = REPO / "data"
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO, "| drive:", DRIVE_ROOT)

In [ ]:
import urllib.request
from fanopt.cfd.phase3 import find_su2
SU2_BIN = find_su2()
if SU2_BIN is None and IN_COLAB:
    LOCAL = Path("/content/su2")
    if not any(LOCAL.rglob("SU2_CFD")):
        zc = DRIVE_ROOT / "su2" / "SU2-v8.0.1-linux64.zip"
        if not zc.exists():
            zc.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(
                "https://github.com/su2code/SU2/releases/download/v8.0.1/SU2-v8.0.1-linux64.zip", str(zc))
        LOCAL.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(zc), "-d", str(LOCAL)], check=True)
    hit = next(LOCAL.rglob("SU2_CFD"), None)
    if hit: subprocess.run(["chmod", "+x", str(hit)], check=False)
    SU2_BIN = str(hit) if hit else None
assert SU2_BIN, "SU2 not found"
print("SU2:", SU2_BIN)

## 2. Campaign config (edit `SESSION_INDEX` per session)

In [ ]:
# ---- EDIT PER SESSION -------------------------------------------------------------------
SESSION_INDEX = 0        # 0 in the FIRST session, 1 in the second, 2 in the third
N_SESSIONS    = 3        # how many Colab sessions you are running in total
SESSION_ID    = f"colab-{SESSION_INDEX}"
# ---- SHARED (identical in every session) ------------------------------------------------
BUDGET     = 300         # stop when the SHARED ledger reaches this many evaluations
N_INIT     = 24          # cold-start Sobol DoE (sliced round-robin across sessions)
N_WORKERS  = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
BATCH_SIZE = N_WORKERS   # match the core count so no worker sits idle (q per session; 3x = total in-flight)
# Fidelity — LOCKED coarse tier from the Stage-2 (2a) probe: coarse preserves the fine ranking
# (Kendall tau=1.0), so the campaign explores on coarse; fine (5,60) is reserved for the 3.C
# top-cluster confirmation. See ADR-0004.
N_CYCLES   = 3           # coarse campaign tier
INNER_ITER = 30
SHARED_DIR = DRIVE_ROOT / "blade_campaign"   # the shared ledger lives here
print(f"session {SESSION_ID} of {N_SESSIONS} | {N_WORKERS} workers | budget {BUDGET} | shared {SHARED_DIR}")

## 3. Run the session (long-running, resumable, async)

In [ ]:
import time
import run_blade_campaign_distributed as campaign
import fanopt.geometry.blade_cad as blade_cad
from fanopt.bo.distributed_campaign import read_ledger
blade_cad.N_RADIAL_SECTIONS = 40  # the objective's geometry resolution (ADR-0004)

CLAIM_TTL = 4 * 3600  # must exceed the coarse per-eval wall time (a live claim must not be stolen)
argv = ["--shared-dir", str(SHARED_DIR), "--session-id", SESSION_ID,
        "--session-index", str(SESSION_INDEX), "--n-sessions", str(N_SESSIONS),
        "--budget", str(BUDGET), "--n-init", str(N_INIT), "--batch-size", str(BATCH_SIZE),
        "--n-workers", str(N_WORKERS), "--su2-bin", SU2_BIN, "--poll-seconds", "10",
        "--claim-ttl", str(CLAIM_TTL)]
if N_CYCLES is not None:   argv += ["--n-cycles", str(N_CYCLES)]
if INNER_ITER is not None: argv += ["--inner-iter", str(INNER_ITER)]
# Long-running + resumable: every eval is appended to the shared Drive ledger, so re-running
# this cell after a Colab drop resumes from the ledger (no work lost).
_n0 = len(read_ledger(SHARED_DIR)[0]); _t0 = time.time()
campaign.main(argv)
_dt_h = (time.time() - _t0) / 3600; _dn = len(read_ledger(SHARED_DIR)[0]) - _n0
print(f"\nthis session: {_dn} evals in {_dt_h:.2f} h"
      + (f"  ->  ~{_dt_h / max(_dn, 1) * N_WORKERS:.2f} h/eval wall" if _dn else ""))

## 4. Validate it ran async (not batched)

In [ ]:
from fanopt.bo.distributed_campaign import validate_async
# Prove the run was async (dispatch-on-completion), not secretly batched. Run this AFTER (or during)
# the campaign — it reads the shared ledger telemetry across ALL sessions.
v = validate_async(SHARED_DIR, N_WORKERS)
print(f"evaluations with telemetry : {v['n_evals']}")
print(f"worker utilization         : {v['utilization']:.0%}")
for s, ps in v["per_session"].items():
    print(f"  session {s}: util={ps['utilization']:.0%}  peak_concurrency={ps['peak_concurrency']}/{N_WORKERS}  n={ps['n_evals']}")
print("==> ASYNC CONFIRMED" if v["is_async"] else "==> WARNING: looks BATCHED / workers idling — investigate")
print(f"\nHow to read it: utilization is the fraction of worker-time actually running an eval.")
print(f"Async keeps every worker busy the instant one frees -> ~100%. A batch loop idles early")
print(f"finishers until their wave-mates complete -> notably lower (~{100*170//206}% at our eval spread).")

## 5. Pareto front across all sessions

In [ ]:
from fanopt.bo.distributed_campaign import pareto_from_ledger, read_ledger
x, y, _ = read_ledger(SHARED_DIR)
front = pareto_from_ledger(SHARED_DIR)
print(f"shared ledger: {len(x)} evaluations across all sessions; {len(front)} on the Pareto front\n")
for d in sorted(front, key=lambda d: -d["j_fan"])[:15]:
    p = d["params"]
    print(f"  J_fan={d['j_fan']:+.3e}  mass={d['mass_kg']*1e3:6.1f} g  "
          f"defl={d['deflection_m']*1e3:6.3f} mm  blades={p['blade_count']}  interp={p['rib_bow_interp']}")